## From Tapology to Topology: Mapping MMA’s Fighter Network

How data scraped from Tapology — one of the largest online MMA databases — can be used to build and visualize a fighter network graph. By turning fight records into network data, we can explore patterns that reveal unexpected connections, influential fighters, and the structure of the MMA world as a living, breathing ecosystem.

In [3]:
import requests
from bs4 import BeautifulSoup

tapology = requests.get('https://www.tapology.com/')
soup = BeautifulSoup(tapology.content, 'html.parser')

print(soup.title.string)

Tapology | MMA & Combat Sports


### Round 1 — The Press Conference: Introducing the Dataset

**Tapology** (*“Tap” — the act of submitting, surrender + “-ology” — from the Greek -λογία (-logia), “the study of"*) is one of the largest online repositories of fighter profiles, fight histories, and event details on the web, and it has become an unofficial encyclopedia of the MMA world. In this project, we model Tapology’s data as a graph where each node corresponds to a fighter, and each edge represents a recorded fight between two fighters. This allows us to construct a network graph capturing the complex interactions within MMA, enabling quantitative analysis of connectivity, centrality, and community structure in the fighter ecosystem.

To build our fighter network, the first step is to scrape data directly from Tapology’s publicly available fight records. Using Python tools like requests and BeautifulSoup, we programmatically navigate fighter profiles to extract key information such as fighter names, fight dates and outcomes. Once collected, the raw data undergoes cleaning and normalization to standardize fighter names and remove duplicate or irrelevant entries. This preparation ensures that when we construct the network graph, the nodes and edges accurately represent meaningful relationships between fighters across different events and weight divisions.

To scrape Tapology without getting blocked, it’s important to rotate User-Agent headers to mimic different browsers.

In [4]:
import httpx
from fake_useragent import UserAgent

ua = UserAgent()

def fetch_page(path: str):
    headers = {"User-Agent": ua.random}
    url = "https://www.tapology.com" + path
    with httpx.Client(headers=headers) as client:
        response = client.get(url)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")
    
do_bronx = fetch_page("/fightcenter/fighters/charles-oliveira-do-bronx")
print(do_bronx.title.string)

Charles Oliveira ("do Bronxs") | MMA Fighter Page | Tapology


To extract meaningful fight data from each fighter’s page, we scrape key information including the fighter’s name, their professional bouts, and details about each fight. For every bout, we capture the opponent’s name and ID, the fight outcome (win/loss), the method of victory or defeat (e.g., KO, submission), and the event where the fight took place. This structured extraction provides the raw material to build edges between fighters and enrich our network with contextual metadata, forming a solid foundation for later analysis.

In [11]:
def scrape_fighter_page(fighter_id):
    soup = fetch_page("/fightcenter/fighters/" + fighter_id)
    bouts = soup.find("div", id="proResults")
    if not bouts:
        print(f"No professional bouts found for fighter ID {fighter_id}")
        return None

    fighter_name = soup.select_one(
        "#fighterPageHeader > div:nth-of-type(2) > div:nth-of-type(1) > div:nth-of-type(2)"
    ).text.strip()

    results = []
    professional_bouts = bouts.find_all("div", {"data-division": "pro"})
    for bout in professional_bouts:
        # The first div contains bout information
        bout_info = bout.find("div")
        sections = bout_info.find_all("div", recursive=False)

        # The second section contains the win/loss method
        method_div = sections[1].find("div")
        if not method_div:
            # Skip cancelled or upcoming bouts
            continue
        method = method_div.get_text(strip=True)
        # The first section contains Win/Loss
        decision = sections[0].get_text(strip=True)

        # The third section contains opponent information
        opponent = sections[2].find("a")
        if not opponent:
            # Skip bouts without an opponent
            continue

        opponent_name = opponent.get_text(strip=True)
        opponent_fighter_id = opponent["href"].split("/")[-1]
        # It also contains event information
        banner = sections[2].find("img")
        event = banner.get("alt", "Unknown Event") if banner else "Unknown Event"

        results.append(
            {
                "fighter_name": fighter_name,
                "opponent_name": opponent_name,
                "decision": decision,
                "method": method,
                "event": event,
                "fighter_id": fighter_id,
                "opponent_fighter_id": opponent_fighter_id,
            }
        )

    return results

bouts = scrape_fighter_page("charles-oliveira-do-bronx")
print(next(iter(bouts)))

{'fighter_name': 'Charles Oliveira', 'opponent_name': 'Ilia Topuria', 'decision': 'L', 'method': 'TKO', 'event': 'UFC', 'fighter_id': 'charles-oliveira-do-bronx', 'opponent_fighter_id': '129278-ilia-topuria'}


To map the interconnected landscape of MMA fighters, we use a **depth-limited Depth-First Search (DFS)** approach to recursively scrape fight data starting from a single fighter. Beginning with a chosen fighter ID, we extract their fight history and then explore their opponents’ profiles, continuing this process up to a specified recursion depth.

This method ensures that we progressively build a network of fighters connected through bouts, while the depth limit and a visited set prevent infinite loops and excessive requests. By applying this controlled traversal, we efficiently capture a meaningful subgraph of the overall MMA fight network, setting the stage for comprehensive network analysis.

In [16]:
def scrape_fighter_network(fighter_id, depth=1, visited=None):
    if visited is None:
        visited = set()
    if depth == 0 or fighter_id in visited:
        return []

    visited.add(fighter_id)
    bouts = scrape_fighter_page(fighter_id)
    if not bouts:
        return []

    network = []
    for bout in bouts:
        opponent_id = bout["opponent_fighter_id"]
        network.append(bout)
        network.extend(scrape_fighter_network(opponent_id, depth - 1, visited))

    return network

charles_bouts = scrape_fighter_network("charles-oliveira-do-bronx", depth=1)
print(f"Charles Oliveira has {len(charles_bouts)} bouts in his MMA career.")

Charles Oliveira has 48 bouts in his MMA career.


Due to time and resource constraints, the recursive scraping depth was limited to 5 levels. This means starting from the chosen fighter, we explore up to five layers of connected opponents in the fight network. For this analysis, I selected Charles Oliveira (“Do Bronx”) as the starting point—not only because he is a top-tier fighter but also my personal favorite. This approach allows us to capture a rich, yet manageable, portion of the MMA network centered around a prominent and active athlete.

In [ ]:
network = scrape_fighter_network("charles-oliveira-do-bronx", depth=5)
print(f"Network size: {len(network)} bouts involving Charles Oliveira and his opponents.")